# Stage 1 — Data Acquisition (Chlorophyll-a Target)

**Region:** Malta–Sicily Channel, 35.5–37.5° N and 12.0–15.5° E  
**Validated period:** 2022-01-01 through 2025-12-31  
**Product:** `cmems_obs-oc_med_bgc-plankton_my_l3-multi-1km_P1D`, Mediterranean multi-sensor L3 daily reprocessed observations at 1 km.

This is the prediction target. Native missing retrievals are retained: no temporal interpolation, gap filling, or substitution with model output is applied. The catalogue version and part are pinned in the configuration cell and checked before download.


In [1]:
MINIMUM_LONGITUDE = 13.90
MAXIMUM_LONGITUDE = 15.20
MINIMUM_LATITUDE =  35.55
MAXIMUM_LATITUDE = 36.90

START = "2022-01-01T00:00:00"
END   = "2025-12-31T00:00:00"


In [2]:
from ipyleaflet import Map, Rectangle, basemaps
from ipywidgets import Layout

m = Map(
    basemap=basemaps.Esri.WorldImagery,
    center=(
        (MINIMUM_LATITUDE + MAXIMUM_LATITUDE) / 2,
        (MINIMUM_LONGITUDE + MAXIMUM_LONGITUDE) / 2,
    ),
    zoom=7,
    layout=Layout(height="500px"),
)

m.add(Rectangle(
    bounds=(
        (MINIMUM_LATITUDE, MINIMUM_LONGITUDE),
        (MAXIMUM_LATITUDE, MAXIMUM_LONGITUDE)
    ), 
    color="blue",
    weight=1,
    fill_opacity=0.15
))

m

Map(center=[36.224999999999994, 14.55], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…

In [3]:
from pathlib import Path
import copernicusmarine

DATASETS = {
    "L3": "cmems_obs-oc_med_bgc-plankton_my_l3-multi-1km_P1D",
    "L4": "cmems_obs-oc_med_bgc-plankton_my_l4-gapfree-multi-1km_P1D",
}

OUT_DIR = Path("data/chl_l3_l4_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)

region_tag = (
    f"{MINIMUM_LONGITUDE}_{MAXIMUM_LONGITUDE}_"
    f"{MINIMUM_LATITUDE}_{MAXIMUM_LATITUDE}"
)

FILES = {
    level: OUT_DIR / f"chl_{level}_{START[:10]}_{END[:10]}_{region_tag}.nc"
    for level in DATASETS
}

for level, dataset_id in DATASETS.items():
    copernicusmarine.subset(
        dataset_id=dataset_id,
        variables=["CHL"],
        start_datetime=START,
        end_datetime=END,
        minimum_latitude=MINIMUM_LATITUDE,
        minimum_longitude=MINIMUM_LONGITUDE,
        maximum_latitude=MAXIMUM_LATITUDE,
        maximum_longitude=MAXIMUM_LONGITUDE,
        output_directory=str(OUT_DIR),
        output_filename=FILES[level].name
    )

INFO - 2026-10-05T17:42:59Z - Selected dataset version: "202411"
INFO - 2026-10-05T17:42:59Z - Selected dataset part: "default"
WARNING - 2026-10-05T17:42:59Z - You are using the dataset cmems_obs-oc_med_bgc-plankton_my_l3-multi-1km_P1D, version '202411', part 'default'. This exact version and part of the dataset will be retired on the 2026-11-24T13:00:00.000Z. For more information you can check: https://marine.copernicus.eu/user-corner/product-roadmap/transition-information


  0%|          | [00:00<?]

INFO - 2026-10-05T17:43:11Z - Total size of the download: 74.92 MB.
INFO - 2026-10-05T17:43:12Z - Selected dataset version: "202311"
INFO - 2026-10-05T17:43:12Z - Selected dataset part: "default"
WARNING - 2026-10-05T17:43:12Z - You are using the dataset cmems_obs-oc_med_bgc-plankton_my_l4-gapfree-multi-1km_P1D, version '202311', part 'default'. This exact version and part of the dataset will be retired on the 2026-11-24T13:00:00.000Z. For more information you can check: https://marine.copernicus.eu/user-corner/product-roadmap/transition-information


  0%|          | [00:00<?]

INFO - 2026-10-05T17:43:26Z - Total size of the download: 74.92 MB.


In [5]:
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display, Markdown


def load_chl(path):
    # xarray decodes scale factors and missing-value markers.
    with xr.open_dataset(path) as ds:
        chl = ds["CHL"].load()

    # Match daily composites by calendar day, regardless of nominal hour
    days = pd.DatetimeIndex(chl.time.values).normalize()
    if days.has_duplicates:
        raise ValueError(f"Multiple CHL fields for the same day in {path}")

    return chl.assign_coords(time=days).sortby("time")

l3 = load_chl(FILES["L3"])
l4 = load_chl(FILES["L4"])

print("L3 units:", l3.attrs.get("units"))
print("L4 units:", l4.attrs.get("units"))
# Both selected products express CHL in mg/m³.


# Stop if dates or pixel coordinates differ. Do not silently regrid.
l3, l4 = xr.align(l3, l4, join="exact")
l3 = l3.transpose("time", "latitude", "longitude")
l4 = l4.transpose(*l3.dims)

valid3 = np.isfinite(l3) & (l3 > 0)
valid4 = np.isfinite(l4) & (l4 > 0)
overlap = valid3 & valid4
filled = (~valid3) & valid4

n_overlap = int(overlap.sum().item())
if n_overlap == 0:
    raise ValueError("No overlapping valid CHL observations")

difference = (l4-l3).where(overlap)

values = difference.values[overlap.values]

# Numerical agreement tolerance, not a scientific accuracy threshold.
close = np.isclose(
    l4.values[overlap.values],
    l3.values[overlap.values],
    rtol=1e-5,
    atol=1e-6,
)

summary = pd.Series({
    "overlapping_valid_pixels": n_overlap,
    "L3_valid_but_L4_missing": int((valid3 & ~valid4).sum().item()),
    "L4_valid_where_L3_missing": int(filled.sum().item()),
    "exactly_equal_pct": 100 * np.mean(values == 0),
    "equal_within_tolerance_pct": 100 * np.mean(close),
    "bias_mg_m3": np.mean(values),
    "MAE_mg_m3": np.mean(np.abs(values)),
    "RMSE_mg_m3": np.sqrt(np.mean(values**2)),
    "maximum_absolute_difference_mg_m3": np.max(np.abs(values))
})

L3 units: milligram m^-3
L4 units: milligram m^-3


In [ ]:
# Counts from the existing summary
n_overlap = int(summary["overlapping_valid_pixels"])
n_missing = int(summary["L3_valid_but_L4_missing"])
n_filled = int(summary["L4_valid_where_L3_missing"])
n_valid_l4 = n_overlap + n_filled

filled_pct = (
    100 * n_filled / n_valid_l4
    if n_valid_l4 > 0 else np.nan
)

# Examine changed observations separately:
# the overall errors include many identical values.
changed_differences = values[
    values != 0
]

n_changed = changed_differences.size
changed_pct = (
    100 * n_changed / n_overlap
    if n_overlap > 0 else np.nan
)

changed_mae = (
    np.mean(np.abs(changed_differences))
    if n_changed else np.nan
)
changed_rmse = (
    np.sqrt(np.mean(changed_differences**2))
    if n_changed else np.nan
)

def number(value, decimals=6):
    return f"{value:,.{decimals}f}" if np.isfinite(value) else "N/A"

rows = [
    ("Valid in both L3 and L4", f"{n_overlap:,}", "pixel-days"),
    ("Valid in L3 but missing in L4", f"{n_missing:,}", "pixel-days"),
    ("Valid in L4 where L3 is missing", f"{n_filled:,}", "pixel-days"),
    ("L4 coverage supplied where L3 is missing", number(filled_pct, 2), "%"),
    ("Exactly equal observations", number(summary["exactly_equal_pct"], 2), "%"),
    ("Equal within numerical tolerance",
     number(summary["equal_within_tolerance_pct"], 2), "%"),
    ("Changed observations", f"{n_changed:,}", "pixel-days "),
    ("Changed observations", number(changed_pct, 2), "%"),
    ("Mean difference: L4 − L3", number(summary["bias_mg_m3"]), "mg/m³"),
    ("Mean absolute difference", number(summary["MAE_mg_m3"]), "mg/m³"),
    ("Root mean square difference", number(summary["RMSE_mg_m3"]), "mg/m³"),
    ("Maximum absolute difference",
     number(summary["maximum_absolute_difference_mg_m3"]), "mg/m³"),
    ("Mean absolute difference — changed pixels only",
     number(changed_mae), "mg/m³"),
    ("Root mean square difference — changed pixels only",
     number(changed_rmse), "mg/m³"),
]

readable_summary = pd.DataFrame(
    rows,
    columns=["Metric", "Value", "Unit"],
).set_index("Metric")

display(readable_summary)

# Explanation generated through f-string interpolation
coverage_text = (
    "Every valid L3 observation also has a valid L4 value."
    if n_missing == 0
    else f"{n_missing:,} valid L3 observations have no valid L4 value."
)

changed_text = (
    f"""Among the **{n_changed:,} changed observations**, the mean absolute
difference is **{changed_mae:.6f} mg/m³** and the root mean square difference
is **{changed_rmse:.6f} mg/m³**. These statistics exclude unchanged values,
which otherwise reduce the overall averages."""
    if n_changed
    else "All overlapping observations are exactly equal."
)

explanation = f"""
Across **{n_overlap:,} overlapping valid pixel-days**,
**{summary['exactly_equal_pct']:.2f}%** of L4 values are exactly equal to L3.
The remaining **{changed_pct:.2f}%** differ. {coverage_text}

L4 supplies **{n_filled:,} additional valid values** where L3 has no valid
observation. These represent **{filled_pct:.2f}% of valid L4 pixel-days**
in this sample.

{changed_text}
"""

display(Markdown(explanation))

,Value,Unit
Metric,,
Valid in both L3 and L4,"10,844,152",pixel-days
Valid in L3 but missing in L4,0,pixel-days
Valid in L4 where L3 is missing,"7,243,028",pixel-days
L4 coverage supplied where L3 is missing,40.05,%
Exactly equal observations,95.41,%
Equal within numerical tolerance,95.45,%
Changed observations,"498,123",pixel-days
Changed observations,4.59,%
Mean difference: L4 − L3,-0.000061,mg/m³



Across **10,844,152 overlapping valid pixel-days**,
**95.41%** of L4 values are exactly equal to L3.
The remaining **4.59%** differ. Every valid L3 observation also has a valid L4 value.

L4 supplies **7,243,028 additional valid values** where L3 has no valid
observation. These represent **40.05% of valid L4 pixel-days**
in this sample.

Among the **498,123 changed observations**, the mean absolute
difference is **0.007622 mg/m³** and the root mean square difference
is **0.027921 mg/m³**. These statistics exclude unchanged values,
which otherwise reduce the overall averages.


**Interpretation:** the equality statistics measure preservation of
available L3 observations, while the additional-coverage count measures
where L4 supplies values without a valid same-day L3 reference.
Differences at observed locations do not imply that L4 ignores L3.
This comparison cannot establish the accuracy of reconstructed values.


In [13]:
DOWNLOADS = {
    "sst": {
        "dataset_id": "cmems_obs-sst_med_phy_my_l3s_P1D-m",
        "dataset_version": "202511",
        "variables": [
            "adjusted_sea_surface_temperature",
        ],
    },
    "wind_ascending": {
        "dataset_id": (
            "cmems_obs-wind_glo_phy_my_l3-"
            "metopb-ascat-asc-0.125deg_P1D-i"
        ),
        "dataset_version": "202311",
        "variables": [
            "eastward_wind",
            "northward_wind",
            "wind_speed",
            "measurement_time",
            "wvc_quality_flag",
        ],
    },
    "wind_descending": {
        "dataset_id": (
            "cmems_obs-wind_glo_phy_my_l3-"
            "metopb-ascat-des-0.125deg_P1D-i"
        ),
        "dataset_version": "202311",
        "variables": [
            "eastward_wind",
            "northward_wind",
            "wind_speed",
            "measurement_time",
            "wvc_quality_flag",
        ],
    },
}

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

COVARIATE_FILES = {}

for name, config in DOWNLOADS.items():
    filename = (
        f"{name}_my_{config['dataset_version']}_"
        f"{START[:10]}_{END[:10]}_{region_tag}.nc"
    )
    path = RAW_DIR / filename
    COVARIATE_FILES[name] = path

    if path.exists():
        print(f"Already exists: {path}")
        continue

    print(f"\nDownloading {name}")

    copernicusmarine.subset(
        dataset_id=config["dataset_id"],
        dataset_version=config["dataset_version"],
        variables=config["variables"],
        start_datetime=START,
        end_datetime=END,
        minimum_longitude=MINIMUM_LONGITUDE,
        maximum_longitude=MAXIMUM_LONGITUDE,
        minimum_latitude=MINIMUM_LATITUDE,
        maximum_latitude=MAXIMUM_LATITUDE,
        output_directory=str(RAW_DIR),
        output_filename=filename,
    )

    if not path.exists():
        raise FileNotFoundError(f"Expected download was not created: {path}")

    print(f"Saved: {path} ({path.stat().st_size/1e6:,.1f} MB)")

print("\nCovariate Files:")
for name, path in COVARIATE_FILES.items():
    print(f"{name} : {path.resolve()}")

INFO - 2026-10-05T17:56:23Z - Selected dataset version: "202511"
INFO - 2026-10-05T17:56:23Z - Selected dataset part: "default"
INFO - 2026-10-05T17:56:28Z - Total size of the download: 1.97 MB.


Saved: data\raw\sst_my_202511_2022-01-01_2025-12-31_13.9_15.2_35.55_36.9.nc (2.1 MB)



INFO - 2026-10-05T17:56:29Z - Selected dataset version: "202311"
INFO - 2026-10-05T17:56:29Z - Selected dataset part: "default"
INFO - 2026-10-05T17:56:33Z - Total size of the download: 3.05 MB.


Saved: data\raw\wind_ascending_my_202311_2022-01-01_2025-12-31_13.9_15.2_35.55_36.9.nc (3.2 MB)



INFO - 2026-10-05T17:56:34Z - Selected dataset version: "202311"
INFO - 2026-10-05T17:56:34Z - Selected dataset part: "default"
INFO - 2026-10-05T17:56:38Z - Total size of the download: 3.05 MB.


Saved: data\raw\wind_descending_my_202311_2022-01-01_2025-12-31_13.9_15.2_35.55_36.9.nc (3.2 MB)

Covariate Files:
sst : C:\GIT\mcast.pages.chloropyll_prediction_v2\expirement_v2\data\raw\sst_my_202511_2022-01-01_2025-12-31_13.9_15.2_35.55_36.9.nc
wind_ascending : C:\GIT\mcast.pages.chloropyll_prediction_v2\expirement_v2\data\raw\wind_ascending_my_202311_2022-01-01_2025-12-31_13.9_15.2_35.55_36.9.nc
wind_descending : C:\GIT\mcast.pages.chloropyll_prediction_v2\expirement_v2\data\raw\wind_descending_my_202311_2022-01-01_2025-12-31_13.9_15.2_35.55_36.9.nc
